# Forecast regional time series

Computes and caches ensemble regional PRECT/FLUT time series aligned with observations, then plots their lead evolution.


## 1. Libraries


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt


## 2. User setup


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")

season = "S0"
experiments = ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0")
variables = ("PRECT", "FLUT")
regions = ("global", "CONUS", "NHMidLat", "Tropics")
max_members = None
force_compute = False
show = True

FIGURE_DPI = 200


## Notes: available selections

Variables: `PRECT` verified against GPM/IMERG and `FLUT` verified against NOAA interpolated OLR. Regions: `global`, `CONUS`, `NHMidLat`, `Tropics`, `NINO3.4`, and `MAR_CONT`. Cached fields include every member, ensemble mean, 25th/75th-percentile envelope, and the aligned observation.

The notebook reads raw daily files only for missing requests. Set `force_compute = True` to rebuild a cache.


## 3. Project setup


In [ ]:
import sys
import matplotlib.pyplot as plt
repo=WORK_DIR.resolve()
if str(repo/"diagnostic") not in sys.path: sys.path.insert(0,str(repo/"diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.forecast_regional_timeseries import REGIONS, VARIABLES, timeseries_cache_path, cache_regional_timeseries, plot_regional_timeseries


## 4. Read, process, and cache


In [ ]:
workflow_name = "fcst_atm_timeseries"
data_dir, figure_dir = workflow_output_dirs(
    workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT
)
registry = experiment_registry(INPUT_DATA_ROOT, season, "fc")
periods = {registry[experiment]["period"] for experiment in experiments}
if len(periods) != 1:
    raise ValueError(f"Selected experiments have different periods: {periods}")
period = periods.pop()

processed = {}
for variable in variables:
    if variable not in ("PRECT", "FLUT"):
        raise ValueError("Observation-comparison time series support PRECT and FLUT")
    for region in regions:
        target = timeseries_cache_path(
            data_dir, variable, region, season, "fc", period
        )
        existed = target.is_file()
        processed[(variable, region)] = cache_regional_timeseries(
            target, data_root=INPUT_DATA_ROOT,
            reference_root=OBSERVATION_ROOT, experiments=experiments,
            season=season, run_segment="fc", variable=variable,
            region=region, period=period, max_members=max_members,
            force_compute=force_compute,
        )
        print("[REUSE]" if existed and not force_compute else "[WRITE]", target)


## 5. Plot, save, and show


In [ ]:
figures = {}
for (variable, region), ds in processed.items():
    fig = plot_regional_timeseries(
        ds, label_map={e: registry[e]["label"] for e in experiments}
    )
    output = figure_dir / f"{variable}_{region}_{season}_forecast_timeseries.png"
    fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
    figures[(variable, region)] = fig
    if not show:
        plt.close(fig)
    print("[FIGURE]", output)
